# Payment Reconciliation Example
**MD Zawwadul Islam | Synthetic portfolio demonstration**

All transactions, bank names and amounts are invented. This is not a bank client engagement.
One source row represents one reported transaction. A reference is intended to identify one transaction.
Both files cover the same sample period and use BDT. Neither file is assumed to be correct when they disagree.

**Goal:** compare two files, retain every source row and amount, and produce an exception list a reviewer can act on.

Open this notebook from the extracted `Reconciliation_Step1` folder. If pandas is missing, run `%pip install pandas` once in a separate cell, then restart the kernel. Run the lesson cells in order with Shift+Enter. Baseline outputs below were executed and checked before delivery.

The notebook regenerates CSV results and a browser-readable HTML report. The included Excel report and screenshots are formatted reference copies of the supplied baseline; they do not refresh automatically when inputs change.

## 1 Set up the project
`Path` locates your files. `Decimal` parses money exactly; the calculation then uses integer paisa (100 paisa = Tk1).
The original CSV files are only read, never overwritten by the reconciliation.
Only pandas is an external Python dependency.

In [1]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import hashlib
import html
import json
import sys
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "reports"
OUT.mkdir(exist_ok=True)
assert (DATA / "synthetic_source_A.csv").is_file(), "Open the notebook from the extracted project folder."
print("Project:", ROOT)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)

Project: /workspace/scratch/3782eaeba416/outputs/reconciliation_step1/Reconciliation_Step1
Python: 3.12.14 | pandas: 2.2.3


## 2 Read and inspect both sources
Read identifiers as text so `000005` keeps its leading zeros. Empty references remain empty strings.
Expect **50 rows in A and 51 in B**. B is deliberately in a different order.
Inspect the bank/date fields, but match on transaction reference only. Dates and bank names are supporting context, not additional matching rules in this example.

In [2]:
raw_a = pd.read_csv(DATA / "synthetic_source_A.csv", dtype=str, keep_default_na=False)
raw_b = pd.read_csv(DATA / "synthetic_source_B.csv", dtype=str, keep_default_na=False)
required = ["transaction_ref", "date", "bank_name", "amount_bdt"]
for label, frame in [("A", raw_a), ("B", raw_b)]:
    assert list(frame.columns) == required, f"Unexpected columns in {label}"
    pd.to_datetime(frame["date"], format="%Y-%m-%d", errors="raise")
    print(f"Source {label}: {len(frame)} rows")
    print(frame.head(5).to_string(index=False))
    print()

Source A: 50 rows
transaction_ref       date      bank_name amount_bdt
         000001 2026-09-01 Example Bank 1    1010.00
         000002 2026-09-02 Example Bank 2    1020.00
         000003 2026-09-03 Example Bank 3    1030.00
         000004 2026-09-04 Example Bank 4    1040.00
         000005 2026-09-05 Example Bank 1    1050.00

Source B: 51 rows
transaction_ref       date      bank_name amount_bdt
         000048 2026-09-06 Example Bank 4    1480.00
         000052 2026-09-14 Example Bank 2    1520.00
         000051 2026-09-14 Example Bank 1    1510.00
         000048 2026-09-06 Example Bank 4    1480.00
                2026-09-05 Example Bank 3    1470.00



## 3 Apply three agreed rules
1. Trim surrounding spaces from references; preserve leading zeros and original values.
2. Compare amounts exactly to the paisa. Difference = **A minus B**, tolerance Tk0.00. Invalid/blank amounts stop the run; they are never changed to zero.
3. Keep missing references and all source rows associated with a repeated reference outside automatic matching. Flag them for review; do not choose a duplicate or guess a match.

Reference `000048` is repeated in B, so its one row in A and two rows in B are all ambiguous. One B row has no reference. A unique reference may still be missing from the other file.

In [3]:
def to_paisa(value):
    try:
        amount = Decimal(str(value).strip())
    except InvalidOperation as exc:
        raise ValueError(f"Invalid amount: {value!r}") from exc
    if not amount.is_finite():
        raise ValueError(f"Invalid amount: {value!r}")
    scaled = amount * 100
    if scaled != scaled.to_integral_value():
        raise ValueError(f"More than two decimal places: {value!r}")
    if abs(scaled) > 9_000_000_000_000_000:
        raise ValueError("Amount exceeds the supported range for this demo")
    return int(scaled)

def sum_paisa(series):
    return sum(int(value) for value in series.dropna())

def prepare(raw):
    frame = raw.copy(deep=True)
    frame["source_row"] = range(2, len(frame) + 2)  # CSV line with header on line 1
    frame["key"] = frame["transaction_ref"].str.strip()
    frame["paisa"] = frame["amount_bdt"].map(to_paisa).astype("Int64")
    return frame

a = prepare(raw_a)
b = prepare(raw_b)
ambiguous_keys = set()
for frame in (a, b):
    repeated = frame["key"].ne("") & frame["key"].duplicated(keep=False)
    ambiguous_keys.update(frame.loc[repeated, "key"])

def partition(frame):
    flagged = frame["key"].eq("") | frame["key"].isin(ambiguous_keys)
    return frame.loc[~flagged].copy(), frame.loc[flagged].copy()

valid_a, flagged_a = partition(a)
valid_b, flagged_b = partition(b)
print("References repeated in either file:", sorted(ambiguous_keys))
print("Rows held for review: A =", len(flagged_a), "; B =", len(flagged_b))
print("Trimmed reference example:", repr(b.loc[b["key"].eq("000005"), "transaction_ref"].iloc[0]), "-> 000005")

References repeated in either file: ['000048']
Rows held for review: A = 1 ; B = 3
Trimmed reference example: ' 000005 ' -> 000005


## 4 Match safe references and retain exceptions
An outer merge retains references found on either side. `validate="one_to_one"` rejects an unexpected duplicate join.
Missing references were excluded before this merge. Repeated-key rows are added back individually with their original amounts and source-row locations.

Expect **55 result rows**: 46 paired rows (representing 92 source records) plus 9 unpaired/flagged source records. A report row and a source record are different units.

In [4]:
joined = valid_a.merge(valid_b, on="key", how="outer", suffixes=("_a", "_b"),
                       indicator=True, validate="one_to_one")
results = pd.DataFrame({
    "key": joined["key"],
    "a_row": joined["source_row_a"].astype("Int64"),
    "b_row": joined["source_row_b"].astype("Int64"),
    "a_paisa": joined["paisa_a"].astype("Int64"),
    "b_paisa": joined["paisa_b"].astype("Int64"),
})
results["difference_paisa"] = results["a_paisa"] - results["b_paisa"]
results["status"] = "Amount difference"
results.loc[joined["_merge"].eq("left_only"), "status"] = "Only in A"
results.loc[joined["_merge"].eq("right_only"), "status"] = "Only in B"
same = joined["_merge"].eq("both") & results["difference_paisa"].eq(0)
results.loc[same, "status"] = "Matched"

held = []
for source, frame in [("a", flagged_a), ("b", flagged_b)]:
    for record in frame.to_dict("records"):
        row = {"key": record["key"], "a_row": pd.NA, "b_row": pd.NA,
               "a_paisa": pd.NA, "b_paisa": pd.NA, "difference_paisa": pd.NA,
               "status": "Missing reference" if record["key"] == "" else "Ambiguous reference"}
        row[f"{source}_row"] = record["source_row"]
        row[f"{source}_paisa"] = record["paisa"]
        held.append(row)

held_frame = pd.DataFrame(held, columns=results.columns)
for col in ["a_row", "b_row", "a_paisa", "b_paisa", "difference_paisa"]:
    held_frame[col] = held_frame[col].astype("Int64")
results = pd.concat([results, held_frame], ignore_index=True)
for col in ["a_row", "b_row", "a_paisa", "b_paisa", "difference_paisa"]:
    results[col] = results[col].astype("Int64")
results = results.sort_values(["key", "status"], na_position="last").reset_index(drop=True)
exceptions = results.loc[results["status"].ne("Matched")].copy()
summary = results["status"].value_counts().rename_axis("Status").reset_index(name="Report rows")
print(summary.to_string(index=False))
print("Exception report rows:", len(exceptions))

             Status  Report rows
            Matched           43
  Amount difference            3
Ambiguous reference            3
          Only in A            3
          Only in B            2
  Missing reference            1
Exception report rows: 12


## 5 Prove no records or money disappeared
A control total is a number used to check completeness, such as a source row count or amount sum.
For each source, the exact set of source-row numbers and the amount total must survive in the results.
These checks do not require A and B to have equal totals. A successful check means the comparison retained the evidence, not that all transactions agree.

In [5]:
def money(paisa):
    return "" if pd.isna(paisa) else f"{Decimal(int(paisa)) / 100:.2f}"

controls = []
for label, frame, prefix in [("A", a, "a"), ("B", b, "b")]:
    source_rows = sorted(int(x) for x in frame["source_row"])
    accounted_rows = sorted(int(x) for x in results[f"{prefix}_row"].dropna())
    assert source_rows == accounted_rows, f"Rows missing or repeated for {label}"
    source_amounts = {int(row): int(amount) for row, amount in zip(frame["source_row"], frame["paisa"])}
    kept = results.loc[results[f"{prefix}_row"].notna()]
    result_amounts = {int(row): int(amount) for row, amount in zip(kept[f"{prefix}_row"], kept[f"{prefix}_paisa"])}
    assert source_amounts == result_amounts, f"A source-row amount changed for {label}"
    source_sum = sum_paisa(frame["paisa"])
    result_sum = sum_paisa(results[f"{prefix}_paisa"])
    assert source_sum == result_sum, f"Amount not conserved for {label}"
    controls.append({"Source": label, "Input rows": len(frame), "Accounted rows": len(accounted_rows),
                     "Input total BDT": money(source_sum), "Accounted total BDT": money(result_sum),
                     "Rows check": "Pass", "Amount check": "Pass"})
controls = pd.DataFrame(controls)
gap_paisa = sum_paisa(a["paisa"]) - sum_paisa(b["paisa"])

# Explain the total gap using disjoint classes, including unresolved rows.
bridge = []
for status, group in results.groupby("status", sort=True):
    impact = sum_paisa(group["a_paisa"]) - sum_paisa(group["b_paisa"])
    bridge.append({"Category": status, "A minus B BDT": money(impact)})
assert sum(to_paisa(row["A minus B BDT"]) for row in bridge) == gap_paisa
bridge = pd.DataFrame(bridge)
print(controls.to_string(index=False))
print("\nTotal gap, A minus B: Tk", money(gap_paisa))
print(bridge.to_string(index=False))

Source  Input rows  Accounted rows Input total BDT Accounted total BDT Rows check Amount check
     A          50              50        62750.00            62750.00       Pass         Pass
     B          51              51        64320.01            64320.01       Pass         Pass

Total gap, A minus B: Tk -1570.01
           Category A minus B BDT
Ambiguous reference      -1480.00
  Amount difference        -50.01
            Matched          0.00
  Missing reference      -1470.00
          Only in A       4460.00
          Only in B      -3030.00


## 6 Check independently specified expected answers
`expected_results.json` records the answers implied by the constructed test cases. It is not created by the matching algorithm.
For the unchanged baseline, check every category and all three known amount differences.
If you deliberately change an input, these original expected answers no longer apply; source-accounting controls still run in step 5.

Manually verify reference `000010`: A is Tk1,100.00, B is Tk1,200.00, so A minus B is **-Tk100.00**.
Verify `000020`: Tk1,200.00 minus Tk1,150.00 = **Tk50.00**.
Verify `000030`: Tk1,300.00 minus Tk1,300.01 = **-Tk0.01**.

In [6]:
expected = json.loads((ROOT / "expected_results.json").read_text(encoding="utf-8"))
current_hashes = {label: hashlib.sha256((DATA / f"synthetic_source_{label}.csv").read_bytes()).hexdigest()
                  for label in ["A", "B"]}
baseline = current_hashes == expected["input_sha256"]
if baseline:
    actual_counts = {str(k): int(v) for k, v in results["status"].value_counts().items()}
    assert actual_counts == expected["status_counts"]
    assert len(a) == expected["source_A_rows"] and len(b) == expected["source_B_rows"]
    assert sum_paisa(a["paisa"]) == expected["source_A_total_paisa"]
    assert sum_paisa(b["paisa"]) == expected["source_B_total_paisa"]
    assert gap_paisa == expected["difference_A_minus_B_paisa"]
    for key, value in expected["known_differences_paisa"].items():
        found = results.loc[results["key"].eq(key), "difference_paisa"]
        assert len(found) == 1 and int(found.iloc[0]) == value
    print("Baseline matches all independently specified expected answers.")
else:
    print("Inputs changed: baseline answers were not applied. Source-accounting checks still passed.")
print("\nKnown difference examples:")
print(results.loc[results["status"].eq("Amount difference")].to_string(index=False))

Baseline matches all independently specified expected answers.

Known difference examples:
   key  a_row  b_row  a_paisa  b_paisa  difference_paisa            status
000010     11     43   110000   120000            -10000 Amount difference
000020     21     33   120000   115000              5000 Amount difference
000030     31     23   130000   130001                -1 Amount difference


## 7 Export the results and read the client report
The CSV files open in Excel. To preserve leading zeros when importing a CSV, use Excel's **Data > From Text/CSV**, set `Reference` to Text, and load. Numeric money columns use plain decimal values in BDT. Empty amounts mean that side is absent, not zero.

`report.html` opens in a browser and contains the summary, source-accounting controls, total-gap explanation and full exception list.
`Reconciliation_Example.xlsx` is the supplied formatted baseline reference; this Python cell does not rewrite it.

For a new Excel delivery after changing inputs, import the newly generated CSV tables into a new workbook and save as `.xlsx`. Do not present the old reference workbook/screenshots as refreshed results.

In [7]:
def present(frame):
    out = frame.copy()
    for col in ["a_paisa", "b_paisa", "difference_paisa"]:
        out[col] = out[col].map(money)
    return out.rename(columns={"key": "Reference", "a_row": "A source row", "b_row": "B source row",
        "a_paisa": "A amount BDT", "b_paisa": "B amount BDT", "difference_paisa": "A minus B BDT", "status": "Status"})

present(results).to_csv(OUT / "all_results.csv", index=False)
present(exceptions).to_csv(OUT / "exceptions.csv", index=False)
summary.to_csv(OUT / "summary.csv", index=False)
controls.to_csv(OUT / "source_controls.csv", index=False)
bridge.to_csv(OUT / "total_gap_bridge.csv", index=False)
for label, frame in [("A", a), ("B", b)]:
    clean = frame.drop(columns="paisa").rename(columns={"key": "clean_reference"})
    clean.to_csv(OUT / f"cleaned_source_{label}.csv", index=False)

status_note = "Unchanged baseline verified." if baseline else "Modified inputs; original baseline answers not applied."
sections = [("Results by status", summary), ("Source-accounting controls", controls),
            ("Explanation of the total gap", bridge), ("Exceptions requiring review", present(exceptions))]
body = "".join(f"<h2>{title}</h2>{frame.to_html(index=False, border=0, na_rep='')}" for title, frame in sections)
page = f"""<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Payment Reconciliation Example</title><style>
body{{font:16px/1.5 Arial,sans-serif;color:#172234;max-width:1120px;margin:40px auto;padding:0 24px;background:#fff}}
h1{{font-size:30px;margin-bottom:4px}}h2{{font-size:20px;margin:30px 0 12px}}
table{{border-collapse:collapse;width:100%;font-size:14px}}th{{background:#20344d;color:#fff;text-align:left}}
th,td{{padding:10px 12px;border-bottom:1px solid #dde3eb}}td{{vertical-align:top}}
p{{max-width:950px}}.muted{{color:#4c5b6d}}.tablewrap{{overflow:auto}}
</style><h1>Payment Reconciliation Example</h1><p class="muted">MD Zawwadul Islam | Synthetic portfolio demonstration</p>
<p>All data and bank names are invented. {html.escape(status_note)}</p>
<p>Match on a trimmed transaction reference; exact paisa comparison; repeated or missing references remain for review.
Difference = A minus B. Missing amounts are not zero. Dates and bank names provide context only.</p>
<p><b>Source total gap: Tk{money(gap_paisa)}.</b> Row and amount controls check completeness, not whether the sources agree.</p>
<div class="tablewrap">{body}</div>
<h2>Review actions</h2><p>Check amount differences against underlying records; investigate one-sided transactions;
resolve repeated references with a reviewer; obtain missing references before assigning a match.</p>
<p class="muted">Exception counts are report rows. A paired difference represents two source records; an ambiguous row represents one source record.</p></html>"""
(OUT / "report.html").write_text(page, encoding="utf-8")
print("Created:", ", ".join(p.name for p in sorted(OUT.iterdir()) if p.suffix in (".csv", ".html")))
print("Open reports/report.html in your browser to inspect the result.")

Created: all_results.csv, cleaned_source_A.csv, cleaned_source_B.csv, exceptions.csv, report.html, source_controls.csv, summary.csv, total_gap_bridge.csv
Open reports/report.html in your browser to inspect the result.


## 8 Finish the portfolio evidence
1. Open the included Excel reference and compare its results with your run.
2. Read `PORTFOLIO_CASE_STUDY.md`. Adapt the wording only after you understand and can explain it.
3. Use the two provided screenshots for the unchanged baseline, or capture your own summary and exceptions after changes.
4. Present the problem, three matching rules, checks, output and review actions. Label the demonstration synthetic.
5. Keep your earlier help for your father as a separate, accurately described experience. Do not identify a bank as a formal client without that relationship and permission.

**Step 1 is complete when:** your run matches the baseline answers, source rows/totals are conserved, you can explain three exception examples, and your case study plus report/screenshots are ready to share. Publishing to your live portfolio is a separate action.

Technical references: [pandas merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) (outer join, null-key warning, one-to-one validation); [Python Decimal](https://docs.python.org/3/library/decimal.html) (exact decimal arithmetic).